In [24]:
import numpy as np
import torch
import pennylane as qml
import pennylane_qiskit
import qiskit
import qiskit_aer

print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("PennyLane:", qml.__version__)
print("Qiskit:", qiskit.__version__)
print("Qiskit Aer:", qiskit_aer.__version__)

NumPy: 1.26.4
Torch: 2.2.2
PennyLane: 0.44.1
Qiskit: 2.2.3
Qiskit Aer: 0.17.2


In [25]:
# ============================================================
# Block 3: Safe 6-qubit noisy backend setup
# ============================================================

import os

# Thread limits reduce the chance of Aer/Jupyter crashes on Mac.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys
import random
import inspect

import numpy as np
import torch
import torch.nn as nn

import pennylane as qml
from pennylane import numpy as pnp

import pennylane_qiskit
import qiskit
import qiskit_aer
from qiskit_aer import AerSimulator
import qiskit_ibm_runtime.fake_provider as fake_provider

# ------------------------------------------------------------
# Reproducibility and dtype
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_default_dtype(torch.float64)

print("Notebook Python:", sys.executable)
print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("PennyLane:", qml.__version__)
print("Qiskit:", qiskit.__version__)
print("Qiskit Aer:", qiskit_aer.__version__)

# ------------------------------------------------------------
# QNN settings
# ------------------------------------------------------------

n_qubits = 6
shots = 512

# Start shallow for the first noisy run.
n_layers = 1
n_ansatz_layers = 1

print("\nQNN settings:")
print("n_qubits:", n_qubits)
print("shots:", shots)
print("n_layers:", n_layers)
print("n_ansatz_layers:", n_ansatz_layers)

# ------------------------------------------------------------
# Select a fake backend with at least 6 qubits
# ------------------------------------------------------------

candidate_backends = []

for name in dir(fake_provider):
    if not name.startswith("Fake"):
        continue

    obj = getattr(fake_provider, name)

    if not inspect.isclass(obj):
        continue

    try:
        backend = obj()
        num_qubits = getattr(backend, "num_qubits", None)

        if num_qubits is not None and num_qubits >= n_qubits:
            candidate_backends.append((name, backend, num_qubits))

    except Exception:
        pass

candidate_backends = sorted(candidate_backends, key=lambda x: x[2])

print("\nAvailable fake backends with at least 6 qubits:")
for name, _, num_qubits in candidate_backends[:10]:
    print(f"{name}: {num_qubits} qubits")

assert len(candidate_backends) > 0, "No fake backend with at least 6 qubits was found."

backend_class_name, fake_backend, backend_num_qubits = candidate_backends[0]

print("\nSelected fake backend:", backend_class_name)
print("Backend name:", fake_backend.name)
print("Backend qubits:", backend_num_qubits)

# ------------------------------------------------------------
# Create noisy Aer simulator
# ------------------------------------------------------------

aer_backend = AerSimulator.from_backend(fake_backend)

aer_backend.set_options(
    max_parallel_threads=1,
    max_parallel_experiments=1,
    max_parallel_shots=1,
)

print("\nThread-limited Aer backend created.")
print(aer_backend)

# ------------------------------------------------------------
# Create PennyLane qiskit.aer device
# ------------------------------------------------------------

dev_noisy = qml.device(
    "qiskit.aer",
    wires=backend_num_qubits,
    backend=aer_backend,
)

print("\nPennyLane noisy device created.")
print("Device wires:", dev_noisy.wires)

Notebook Python: /Users/mansigoel/Documents/GitHub/Quantum-Machine-Learning-/venv/bin/python
NumPy: 1.26.4
Torch: 2.2.2
PennyLane: 0.44.1
Qiskit: 2.2.3
Qiskit Aer: 0.17.2

QNN settings:
n_qubits: 6
shots: 512
n_layers: 1
n_ansatz_layers: 1

Available fake backends with at least 6 qubits:
FakeCasablancaV2: 7 qubits
FakeJakartaV2: 7 qubits
FakeLagosV2: 7 qubits
FakeNairobiV2: 7 qubits
FakeOslo: 7 qubits
FakePerth: 7 qubits
FakeMelbourneV2: 15 qubits
FakeGuadalupeV2: 16 qubits
FakeAlmadenV2: 20 qubits
FakeBoeblingenV2: 20 qubits

Selected fake backend: FakeCasablancaV2
Backend name: fake_casablanca
Backend qubits: 7

Thread-limited Aer backend created.
AerSimulator('aer_simulator_from(fake_casablanca)'
             noise_model=<NoiseModel on ['x', 'cx', 'measure', 'sx', 'reset', 'id']>)

PennyLane noisy device created.
Device wires: Wires([0, 1, 2, 3, 4, 5, 6])


In [26]:
# ============================================================
# Block 3B: Use fake-backend noise model without coupling-map constraints
# ============================================================

from qiskit_aer.noise import NoiseModel

# ------------------------------------------------------------
# Build a noise model from the selected fake backend
# ------------------------------------------------------------

noise_model = NoiseModel.from_backend(fake_backend)

print("Noise model created from:", fake_backend.name)
print("Noise basis gates:", noise_model.basis_gates)

# ------------------------------------------------------------
# Create Aer simulator with noise, but without hardware coupling constraints
# ------------------------------------------------------------
# Key difference:
#   We are NOT using AerSimulator.from_backend(fake_backend)
# because that also carries device connectivity/coupling constraints.
#
# Instead, we use only the noise model.

aer_backend = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    seed_simulator=SEED,
)

aer_backend.set_options(
    max_parallel_threads=1,
    max_parallel_experiments=1,
    max_parallel_shots=1,
)

print("\nAer simulator created with noise model only.")
print(aer_backend)

# ------------------------------------------------------------
# Recreate PennyLane device
# ------------------------------------------------------------
# Now use exactly n_qubits wires, not all backend wires.
# This avoids weird physical-wire remapping like logical wire 5 -> physical wire 6.

dev_noisy = qml.device(
    "qiskit.aer",
    wires=n_qubits,
    backend=aer_backend,
    basis_gates=noise_model.basis_gates,
    optimization_level=1,
    seed_transpiler=SEED,
)

print("\nPennyLane noisy device recreated.")
print("Device wires:", dev_noisy.wires)
print("Shots will be set on QNode:", shots)

Noise model created from: fake_casablanca
Noise basis gates: ['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']

Aer simulator created with noise model only.
AerSimulator('aer_simulator'
             noise_model=<NoiseModel on ['x', 'cx', 'measure', 'sx', 'reset', 'id']>)

PennyLane noisy device recreated.
Device wires: Wires([0, 1, 2, 3, 4, 5])
Shots will be set on QNode: 512


In [27]:
# ============================================================
# Block 4: Define and test the noisy 6-qubit QNN circuit
# ============================================================

@qml.set_shots(shots)
@qml.qnode(dev_noisy, interface=None, diff_method=None)
def circuit_parallel_noisy(q_params, x):
    """
    Noisy 6-qubit data re-uploading circuit.

    Args:
        q_params:
            Shape = (n_layers, n_ansatz_layers, n_qubits, 3)

        x:
            Shape = (n_qubits,)

    Returns:
        List of 6 expectation values:
            [<Z0>, <Z1>, ..., <Z5>]
    """

    for layer in range(n_layers):

        # Data upload
        for q in range(n_qubits):
            qml.RY(x[q], wires=q)

        # Trainable ansatz
        for ansatz_layer in range(n_ansatz_layers):

            for q in range(n_qubits):
                qml.Rot(
                    q_params[layer, ansatz_layer, q, 0],
                    q_params[layer, ansatz_layer, q, 1],
                    q_params[layer, ansatz_layer, q, 2],
                    wires=q,
                )

            # Logical nearest-neighbour CNOT chain
            for q in range(n_qubits - 1):
                qml.CNOT(wires=[q, q + 1])

    return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]


test_q_params = 0.01 * np.random.randn(
    n_layers,
    n_ansatz_layers,
    n_qubits,
    3,
)

test_x = np.random.randn(n_qubits)

test_q_out = circuit_parallel_noisy(test_q_params, test_x)

print("Noisy QNN test call successful.")
print("test_x shape:", test_x.shape)
print("test_q_params shape:", test_q_params.shape)
print("Quantum output:", test_q_out)
print("Quantum output length:", len(test_q_out))

Noisy QNN test call successful.
test_x shape: (6,)
test_q_params shape: (1, 1, 6, 3)
Quantum output: [array(0.59375), array(0.09765625), array(0.01953125), array(-0.01171875), array(-0.015625), array(0.04296875)]
Quantum output length: 6


/Users/mansigoel/Documents/GitHub/Quantum-Machine-Learning-/venv/lib/python3.11/site-packages/qiskit/compiler/transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


In [28]:
# ============================================================
# Block 5: Define Hybrid QNN model without torch.Tensor.numpy()
# ============================================================

class HybridModelNoisy(nn.Module):
    def __init__(self):
        super().__init__()

        # Quantum parameters:
        # shape = (n_layers, n_ansatz_layers, n_qubits, 3)
        self.q_params = nn.Parameter(
            0.01 * torch.randn(
                n_layers,
                n_ansatz_layers,
                n_qubits,
                3,
                dtype=torch.float64,
            )
        )

        # Classical head:
        # takes 6 quantum expectation values and maps to 1 output
        self.classical = nn.Sequential(
            nn.Linear(n_qubits, 1),
            nn.Tanh(),
        ).double()

    def _torch_to_numpy_safe(self, tensor):
        """
        Converts a torch tensor to NumPy without calling tensor.numpy().

        This avoids the PyTorch NumPy bridge issue:
            RuntimeError: Numpy is not available
        """

        if isinstance(tensor, torch.Tensor):
            return np.asarray(
                tensor.detach().cpu().tolist(),
                dtype=np.float64,
            )

        return np.asarray(tensor, dtype=np.float64)

    def quantum_features(self, x_batch, q_params=None):
        """
        Evaluates the noisy QNN one sample at a time.

        Args:
            x_batch:
                torch tensor of shape (batch_size, n_qubits)

            q_params:
                optional q_params candidate.
                Used during SPSA when evaluating theta + cDelta
                and theta - cDelta.

        Returns:
            torch tensor of shape (batch_size, n_qubits)
        """

        if q_params is None:
            q_params = self.q_params

        q_params_np = self._torch_to_numpy_safe(q_params)

        q_outputs = []

        for x in x_batch:
            x_np = self._torch_to_numpy_safe(x)

            q_out = circuit_parallel_noisy(q_params_np, x_np)

            # q_out is a list of scalar arrays.
            q_out_list = [float(v) for v in q_out]

            q_out_tensor = torch.tensor(
                q_out_list,
                dtype=torch.float64,
                device=x_batch.device,
            )

            q_outputs.append(q_out_tensor)

        q_outputs = torch.stack(q_outputs)

        return q_outputs

    def forward(self, x_batch, q_params=None):
        """
        Full hybrid forward pass:

            x_batch -> noisy QNN -> quantum features -> classical head
        """

        q_outputs = self.quantum_features(
            x_batch,
            q_params=q_params,
        )

        # Important:
        # no backprop through noisy quantum circuit
        q_outputs = q_outputs.detach()

        y_pred = self.classical(q_outputs)

        return y_pred.squeeze(-1)

In [29]:
# ============================================================
# Block 6: Test HybridModelNoisy forward pass
# ============================================================

model = HybridModelNoisy().double()

print(model)

batch_size = 3

X_dummy = torch.randn(
    batch_size,
    n_qubits,
    dtype=torch.float64,
)

y_dummy = torch.randn(
    batch_size,
    dtype=torch.float64,
)

y_pred_dummy = model(X_dummy)

print("\nX_dummy shape:", X_dummy.shape)
print("y_dummy shape:", y_dummy.shape)
print("y_pred_dummy shape:", y_pred_dummy.shape)
print("y_pred_dummy:", y_pred_dummy)

HybridModelNoisy(
  (classical): Sequential(
    (0): Linear(in_features=6, out_features=1, bias=True)
    (1): Tanh()
  )
)

X_dummy shape: torch.Size([3, 6])
y_dummy shape: torch.Size([3])
y_pred_dummy shape: torch.Size([3])
y_pred_dummy: tensor([0.3469, 0.3541, 0.3620], grad_fn=<SqueezeBackward1>)


In [30]:
# ============================================================
# Block 7: Corrected SPSA + Adam training utilities
# ============================================================

# ------------------------------------------------------------
# Loss function
# ------------------------------------------------------------
# Current model ends with Tanh(), so this assumes regression-style
# targets scaled roughly to [-1, 1].
#
# If later you use binary classification, we should remove Tanh()
# and use BCEWithLogitsLoss instead.

loss_fn = nn.MSELoss()


# ------------------------------------------------------------
# PennyLane SPSA optimizer for q_params
# ------------------------------------------------------------

spsa_opt = qml.SPSAOptimizer(
    maxiter=300,
    a=0.02,
    c=0.05,
    A=50,
    alpha=0.602,
    gamma=0.101,
)


# ------------------------------------------------------------
# PyTorch Adam optimizer for classical head
# ------------------------------------------------------------

head_optimizer = torch.optim.Adam(
    model.classical.parameters(),
    lr=1e-3,
)


# ------------------------------------------------------------
# Helper: initialize q_params for PennyLane SPSA
# ------------------------------------------------------------
# We avoid torch_tensor.numpy() because your current PyTorch setup
# had NumPy bridge issues. So we use .tolist().

def init_q_params_for_spsa(model):
    q_params_list = model.q_params.detach().cpu().tolist()

    q_params_pl = pnp.array(
        q_params_list,
        dtype=np.float64,
        requires_grad=True,
    )

    return q_params_pl


q_params_pl = init_q_params_for_spsa(model)

print("Initialized q_params_pl for PennyLane SPSA.")
print("q_params_pl shape:", q_params_pl.shape)


# ------------------------------------------------------------
# Helper: copy PennyLane q_params back into PyTorch model
# ------------------------------------------------------------

def copy_q_params_to_model(model, q_params_pl):
    q_params_np = np.asarray(
        q_params_pl,
        dtype=np.float64,
    )

    q_params_tensor = torch.tensor(
        q_params_np.tolist(),
        dtype=torch.float64,
        device=model.q_params.device,
    )

    with torch.no_grad():
        model.q_params.copy_(q_params_tensor)


# ------------------------------------------------------------
# SPSA cost function factory
# ------------------------------------------------------------
# This creates:
#
#     cost(q_params_candidate) -> PennyLane/NumPy scalar
#
# Important:
#     Do NOT return a Python float.
#     PennyLane SPSA expects the cost value to behave like a
#     NumPy/PennyLane scalar with attributes like .size.

def make_spsa_cost(model, X_batch, y_batch):
    """
    Creates a cost function of q_params only.

    During this cost evaluation:
        - q_params are provided by PennyLane SPSA
        - classical head is held fixed
        - full hybrid loss is evaluated
    """

    X_batch = X_batch.detach()
    y_batch = y_batch.detach()

    def cost(q_params_candidate):
        model.eval()

        with torch.no_grad():
            y_pred = model(
                X_batch,
                q_params=q_params_candidate,
            )

            loss = loss_fn(
                y_pred,
                y_batch,
            )

        # Return PennyLane-compatible scalar, not Python float.
        return pnp.array(
            loss.detach().cpu().item(),
            requires_grad=False,
        )

    return cost


# ------------------------------------------------------------
# One PennyLane SPSA step for q_params
# ------------------------------------------------------------

def pennylane_spsa_step_qparams(
    model,
    q_params_pl,
    X_batch,
    y_batch,
):
    """
    Performs one SPSA update on q_params only.

    Classical head parameters are fixed during this step.
    """

    cost_fn = make_spsa_cost(
        model,
        X_batch,
        y_batch,
    )

    q_params_pl_new = spsa_opt.step(
        cost_fn,
        q_params_pl,
    )

    copy_q_params_to_model(
        model,
        q_params_pl_new,
    )

    return q_params_pl_new


# ------------------------------------------------------------
# One Adam step for classical head
# ------------------------------------------------------------

def adam_step_classical_head(
    model,
    X_batch,
    y_batch,
):
    """
    Performs one Adam update on the classical head only.

    q_params are effectively fixed because quantum features are detached
    inside model.forward().
    """

    model.train()

    head_optimizer.zero_grad()

    y_pred = model(X_batch)

    loss = loss_fn(
        y_pred,
        y_batch,
    )

    loss.backward()

    head_optimizer.step()

    return float(loss.detach().cpu().item())


# ------------------------------------------------------------
# Diagnostic loss evaluation
# ------------------------------------------------------------

def evaluate_hybrid_loss(
    model,
    X_batch,
    y_batch,
):
    model.eval()

    with torch.no_grad():
        y_pred = model(X_batch)

        loss = loss_fn(
            y_pred,
            y_batch,
        )

    return float(loss.detach().cpu().item())


# ------------------------------------------------------------
# Quick test: check SPSA cost output type
# ------------------------------------------------------------

X_test_small = torch.randn(
    1,
    n_qubits,
    dtype=torch.float64,
)

y_test_small = torch.tanh(
    torch.randn(
        1,
        dtype=torch.float64,
    )
)

cost_test = make_spsa_cost(
    model,
    X_test_small,
    y_test_small,
)

cost_value = cost_test(q_params_pl)

print("\nSPSA cost test:")
print("Cost value:", cost_value)
print("Type:", type(cost_value))
print("Shape:", getattr(cost_value, "shape", None))
print("Size:", getattr(cost_value, "size", None))

print("\nCorrected SPSA + Adam utility functions defined.")

Initialized q_params_pl for PennyLane SPSA.
q_params_pl shape: (1, 1, 6, 3)

SPSA cost test:
Cost value: 0.17690812719444907
Type: <class 'pennylane.numpy.tensor.tensor'>
Shape: ()
Size: 1

Corrected SPSA + Adam utility functions defined.


In [31]:
# ============================================================
# Block 8A: Faster one SPSA + Adam sanity check
# ============================================================

num_dummy_samples = 3

X_train_dummy = torch.randn(
    num_dummy_samples,
    n_qubits,
    dtype=torch.float64,
)

y_train_dummy = torch.tanh(
    torch.randn(
        num_dummy_samples,
        dtype=torch.float64,
    )
)

batch_size = 1

idx = torch.randint(
    low=0,
    high=X_train_dummy.shape[0],
    size=(batch_size,),
)

X_batch = X_train_dummy[idx]
y_batch = y_train_dummy[idx]

print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)

loss_before = evaluate_hybrid_loss(
    model,
    X_batch,
    y_batch,
)

print("\nLoss before update:", loss_before)

q_norm_before = torch.norm(model.q_params.detach()).item()
head_weight_norm_before = torch.norm(model.classical[0].weight.detach()).item()
head_bias_norm_before = torch.norm(model.classical[0].bias.detach()).item()

print("\nBefore update:")
print("q_params norm:", q_norm_before)
print("head weight norm:", head_weight_norm_before)
print("head bias norm:", head_bias_norm_before)

# ------------------------------------------------------------
# 1. SPSA update for quantum parameters
# ------------------------------------------------------------

print("\nStarting SPSA update...")

q_params_pl = pennylane_spsa_step_qparams(
    model,
    q_params_pl,
    X_batch,
    y_batch,
)

print("SPSA update completed.")

# ------------------------------------------------------------
# 2. Adam update for classical head
# ------------------------------------------------------------

print("\nStarting Adam update...")

head_loss = adam_step_classical_head(
    model,
    X_batch,
    y_batch,
)

print("Adam update completed.")
print("Adam/head loss:", head_loss)

q_norm_after = torch.norm(model.q_params.detach()).item()
head_weight_norm_after = torch.norm(model.classical[0].weight.detach()).item()
head_bias_norm_after = torch.norm(model.classical[0].bias.detach()).item()

print("\nAfter update:")
print("q_params norm:", q_norm_after)
print("head weight norm:", head_weight_norm_after)
print("head bias norm:", head_bias_norm_after)

print("\nChanges:")
print("q_params norm change:", q_norm_after - q_norm_before)
print("head weight norm change:", head_weight_norm_after - head_weight_norm_before)
print("head bias norm change:", head_bias_norm_after - head_bias_norm_before)

X_batch shape: torch.Size([1, 6])
y_batch shape: torch.Size([1])

Loss before update: 1.02699987939799

Before update:
q_params norm: 0.0270951572285556
head weight norm: 0.5189621352805543
head bias norm: 0.35460349035422717

Starting SPSA update...
SPSA update completed.

Starting Adam update...
Adam update completed.
Adam/head loss: 1.0269146315610667

After update:
q_params norm: 0.02762250856206953
head weight norm: 0.5192270920677889
head bias norm: 0.35360349036045163

Changes:
q_params norm change: 0.0005273513335139306
head weight norm change: 0.0002649567872345937
head bias norm change: -0.0009999999937755355


In [32]:
# ============================================================
# Block 9: Tiny multi-step SPSA + Adam training loop
# ============================================================

def train_spsa_adam_loop(
    model,
    q_params_pl,
    X_train,
    y_train,
    num_iters=5,
    batch_size=1,
    log_every=1,
):
    """
    Tiny SPSA + Adam training loop.

    Each iteration:
        1. Sample a mini-batch
        2. Update q_params using PennyLane SPSA
        3. Update classical head using PyTorch Adam
        4. Optionally evaluate and log current loss

    Returns:
        updated q_params_pl
        history dictionary
    """

    history = {
        "iter": [],
        "loss": [],
        "q_norm": [],
        "head_weight_norm": [],
        "head_bias_norm": [],
    }

    n_samples = X_train.shape[0]

    for k in range(num_iters):
        # ----------------------------------------------------
        # Sample mini-batch
        # ----------------------------------------------------
        idx = torch.randint(
            low=0,
            high=n_samples,
            size=(batch_size,),
        )

        X_batch = X_train[idx]
        y_batch = y_train[idx]

        # ----------------------------------------------------
        # 1. SPSA update for quantum parameters
        # ----------------------------------------------------
        q_params_pl = pennylane_spsa_step_qparams(
            model,
            q_params_pl,
            X_batch,
            y_batch,
        )

        # ----------------------------------------------------
        # 2. Adam update for classical head
        # ----------------------------------------------------
        head_loss = adam_step_classical_head(
            model,
            X_batch,
            y_batch,
        )

        # ----------------------------------------------------
        # Logging
        # ----------------------------------------------------
        if k % log_every == 0:
            # Use the same batch for logging to keep it cheap.
            # This is noisy, but good enough for debugging.
            current_loss = evaluate_hybrid_loss(
                model,
                X_batch,
                y_batch,
            )

            q_norm = torch.norm(model.q_params.detach()).item()
            head_weight_norm = torch.norm(model.classical[0].weight.detach()).item()
            head_bias_norm = torch.norm(model.classical[0].bias.detach()).item()

            history["iter"].append(k)
            history["loss"].append(current_loss)
            history["q_norm"].append(q_norm)
            history["head_weight_norm"].append(head_weight_norm)
            history["head_bias_norm"].append(head_bias_norm)

            print(
                f"iter={k:03d} | "
                f"loss={current_loss:.6f} | "
                f"head_loss={head_loss:.6f} | "
                f"q_norm={q_norm:.6f} | "
                f"head_w_norm={head_weight_norm:.6f} | "
                f"head_b_norm={head_bias_norm:.6f}"
            )

    return q_params_pl, history

In [33]:
# ============================================================
# Block 9A: Run tiny dummy SPSA + Adam training loop
# ============================================================

num_dummy_samples = 8

X_train_dummy = torch.randn(
    num_dummy_samples,
    n_qubits,
    dtype=torch.float64,
)

y_train_dummy = torch.tanh(
    torch.randn(
        num_dummy_samples,
        dtype=torch.float64,
    )
)

print("X_train_dummy shape:", X_train_dummy.shape)
print("y_train_dummy shape:", y_train_dummy.shape)

q_params_pl, dummy_history = train_spsa_adam_loop(
    model=model,
    q_params_pl=q_params_pl,
    X_train=X_train_dummy,
    y_train=y_train_dummy,
    num_iters=5,
    batch_size=1,
    log_every=1,
)

print("\nTiny dummy training loop completed.")
print("History keys:", dummy_history.keys())

X_train_dummy shape: torch.Size([8, 6])
y_train_dummy shape: torch.Size([8])
iter=000 | loss=0.009982 | head_loss=0.009691 | q_norm=0.027801 | head_w_norm=0.519372 | head_b_norm=0.352998
iter=001 | loss=0.361130 | head_loss=0.361760 | q_norm=0.026867 | head_w_norm=0.518991 | head_b_norm=0.352947
iter=002 | loss=0.115609 | head_loss=0.116938 | q_norm=0.026838 | head_w_norm=0.518579 | head_b_norm=0.353088
iter=003 | loss=0.113940 | head_loss=0.115609 | q_norm=0.026824 | head_w_norm=0.518170 | head_b_norm=0.353363
iter=004 | loss=1.593133 | head_loss=1.593505 | q_norm=0.027760 | head_w_norm=0.517898 | head_b_norm=0.353140

Tiny dummy training loop completed.
History keys: dict_keys(['iter', 'loss', 'q_norm', 'head_weight_norm', 'head_bias_norm'])


In [34]:
# ============================================================
# Block 10: Load LSTM-AE latents safely for fixed noisy QNN
# ============================================================

from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

project_root = Path("/Users/mansigoel/Documents/GitHub/Quantum-Machine-Learning-")
data_reupload_path = project_root / "data-re upload"

if str(data_reupload_path) not in sys.path:
    sys.path.append(str(data_reupload_path))

from models.LSTMAE_pipeline import (
    load_lstm_ae_checkpoint,
    extract_series,
    series_to_Xy,
)

# ------------------------------------------------------------
# Fixed noisy QNN settings
# ------------------------------------------------------------

fixed_n_qubits = 6
fixed_n_layers = n_layers
fixed_n_ansatz_layers = n_ansatz_layers

print("Fixed noisy QNN configuration:")
print("fixed_n_qubits:", fixed_n_qubits)
print("fixed_n_layers:", fixed_n_layers)
print("fixed_n_ansatz_layers:", fixed_n_ansatz_layers)

assert n_qubits == fixed_n_qubits, (
    f"Current noisy circuit has n_qubits={n_qubits}, "
    f"but this data block expects {fixed_n_qubits}."
)

# ------------------------------------------------------------
# Locate and load SPEI CSV
# ------------------------------------------------------------

csv_candidates = [
    project_root / "SPEI_AllScales_Napak - SPEI_AllScales_Napak.csv",
    data_reupload_path / "SPEI_AllScales_Napak - SPEI_AllScales_Napak.csv",
    Path.cwd() / "SPEI_AllScales_Napak - SPEI_AllScales_Napak.csv",
]

file_path = None

for candidate in csv_candidates:
    if candidate.exists():
        file_path = candidate
        break

assert file_path is not None, (
    "Could not find SPEI_AllScales_Napak - SPEI_AllScales_Napak.csv "
    "in project_root, data-re upload, or current working directory."
)

print("\nUsing CSV file:")
print(file_path)

df = pd.read_csv(file_path)

train_end_idx = 434

df_train = df.iloc[:train_end_idx].copy()
df_test = df.iloc[train_end_idx:].copy()

print("\nRaw data shapes:")
print("df:", df.shape)
print("df_train:", df_train.shape)
print("df_test:", df_test.shape)

# ------------------------------------------------------------
# Locate and load pretrained LSTM-AE checkpoint
# ------------------------------------------------------------

checkpoint_dir = (
    project_root
    / "data-re upload"
    / "experiments"
    / "lstm_ae_tuning_latent6_window20"
)

print("\nCheckpoint directory:")
print(checkpoint_dir)
print("Exists?", checkpoint_dir.exists())

assert checkpoint_dir.exists(), (
    f"Checkpoint directory not found: {checkpoint_dir}"
)

available_checkpoints = sorted(checkpoint_dir.glob("*.pth"))

print("\nAvailable checkpoints:")
for ckpt in available_checkpoints:
    print(" -", ckpt.name)

assert len(available_checkpoints) > 0, (
    f"No .pth checkpoint files found in {checkpoint_dir}"
)

preferred_checkpoint_name = "run_001_lr_0.01_bs_16_latent_6_win_20.pth"
preferred_checkpoint_path = checkpoint_dir / preferred_checkpoint_name

if preferred_checkpoint_path.exists():
    best_ae_checkpoint_path = preferred_checkpoint_path
else:
    best_ae_checkpoint_path = available_checkpoints[0]

print("\nUsing checkpoint:")
print(best_ae_checkpoint_path)

ae_checkpoint = load_lstm_ae_checkpoint(
    checkpoint_path=str(best_ae_checkpoint_path),
    device="cpu",
)

best_model = ae_checkpoint["model"]
best_encoder = ae_checkpoint["encoder"]
best_scaler = ae_checkpoint["scaler"]
best_config = ae_checkpoint["config"]

print("\nLoaded LSTM-AE checkpoint config:")
print(best_config)

# ------------------------------------------------------------
# Create train/validation windows from training portion
# ------------------------------------------------------------

value_col = best_config.value_col
window_size = best_config.window_size
train_window_end = best_config.train_window_end

print("\nLSTM-AE config values:")
print("value_col:", value_col)
print("window_size:", window_size)
print("train_window_end:", train_window_end)

train_series = extract_series(
    df_train,
    value_col,
)

train_series_scaled = best_scaler.transform(
    train_series
).astype(np.float32)

X_all, y_all = series_to_Xy(
    train_series_scaled,
    window_size,
)

X_lstm_train = X_all[:train_window_end]
y_lstm_train = y_all[:train_window_end]

X_lstm_val = X_all[train_window_end:]
y_lstm_val = y_all[train_window_end:]

print("\nLSTM train/val window data:")
print("X_all:", X_all.shape)
print("y_all:", y_all.shape)
print("X_lstm_train:", X_lstm_train.shape)
print("y_lstm_train:", y_lstm_train.shape)
print("X_lstm_val:", X_lstm_val.shape)
print("y_lstm_val:", y_lstm_val.shape)

# ------------------------------------------------------------
# Create test windows from held-out test portion
# ------------------------------------------------------------

test_series = extract_series(
    df_test,
    value_col,
)

test_series_scaled = best_scaler.transform(
    test_series
).astype(np.float32)

X_lstm_test, y_lstm_test = series_to_Xy(
    test_series_scaled,
    window_size,
)

print("\nLSTM test window data:")
print("X_lstm_test:", X_lstm_test.shape)
print("y_lstm_test:", y_lstm_test.shape)

# ------------------------------------------------------------
# Safe latent extraction from pretrained LSTM-AE
# ------------------------------------------------------------
# Important:
#     The LSTM-AE was trained in float32.
#     Our noisy QNN uses float64 later.
#
# We therefore extract latents using float32, then convert the final
# QNN tensors to float64 afterward.
#
# This avoids:
#     RuntimeError: get_mkldnn_dtype: unsupported data type

latent_device = "cpu"

def extract_latents_safe(
    ae_model,
    X,
    device="cpu",
    batch_size=128,
):
    """
    Safe latent extraction for pretrained LSTM-AE.

    Forces:
        - model.float()
        - input float32
        - MKLDNN disabled during LSTM forward
        - no torch.Tensor.numpy() dependency
    """

    device = torch.device(device)

    ae_model = ae_model.to(device)
    ae_model = ae_model.float()
    ae_model.eval()

    X_np = np.asarray(X, dtype=np.float32)

    latents = []

    old_mkldnn_state = torch.backends.mkldnn.enabled
    torch.backends.mkldnn.enabled = False

    try:
        with torch.no_grad():
            for start in range(0, X_np.shape[0], batch_size):
                end = start + batch_size

                X_batch_np = X_np[start:end]

                X_batch = torch.tensor(
                    X_batch_np.tolist(),
                    dtype=torch.float32,
                    device=device,
                )

                x_enc_repeated, _ = ae_model.encoder(X_batch)

                z_batch = x_enc_repeated[:, 0, :]

                z_batch_np = np.asarray(
                    z_batch.detach().cpu().tolist(),
                    dtype=np.float32,
                )

                latents.append(z_batch_np)

    finally:
        torch.backends.mkldnn.enabled = old_mkldnn_state

    z = np.concatenate(latents, axis=0)

    return z


z_train = extract_latents_safe(
    ae_model=best_model,
    X=X_lstm_train,
    device=latent_device,
    batch_size=128,
)

z_val = extract_latents_safe(
    ae_model=best_model,
    X=X_lstm_val,
    device=latent_device,
    batch_size=128,
)

z_test = extract_latents_safe(
    ae_model=best_model,
    X=X_lstm_test,
    device=latent_device,
    batch_size=128,
)

print("\nExtracted latent vectors:")
print("z_train:", z_train.shape, z_train.dtype)
print("z_val:", z_val.shape, z_val.dtype)
print("z_test:", z_test.shape, z_test.dtype)

assert z_train.ndim == 2, (
    f"Expected z_train to be 2D, got shape {z_train.shape}"
)

assert z_train.shape[1] == fixed_n_qubits, (
    f"Latent dimension is {z_train.shape[1]}, "
    f"but noisy QNN expects {fixed_n_qubits} qubits/features."
)

# ------------------------------------------------------------
# Convert latents to QNN input angles
# ------------------------------------------------------------
# This matches your classical QNN simulation notebook:
#     X_qnn = pi * z

X_qnn_train = np.pi * z_train
X_qnn_val = np.pi * z_val
X_qnn_test = np.pi * z_test

y_qnn_train = y_lstm_train.reshape(-1)
y_qnn_val = y_lstm_val.reshape(-1)
y_qnn_test = y_lstm_test.reshape(-1)

print("\nQNN angle data:")
print("X_qnn_train:", X_qnn_train.shape)
print("X_qnn_val:", X_qnn_val.shape)
print("X_qnn_test:", X_qnn_test.shape)
print("y_qnn_train:", y_qnn_train.shape)
print("y_qnn_val:", y_qnn_val.shape)
print("y_qnn_test:", y_qnn_test.shape)

# ------------------------------------------------------------
# Angle diagnostics
# ------------------------------------------------------------

print("\nAngle diagnostics:")
print("Train angle min:", X_qnn_train.min())
print("Train angle max:", X_qnn_train.max())
print("Val angle min:", X_qnn_val.min())
print("Val angle max:", X_qnn_val.max())
print("Test angle min:", X_qnn_test.min())
print("Test angle max:", X_qnn_test.max())

print(
    "Train fraction |theta| > pi:",
    np.mean(np.abs(X_qnn_train.flatten()) > np.pi),
)
print(
    "Val fraction |theta| > pi:",
    np.mean(np.abs(X_qnn_val.flatten()) > np.pi),
)
print(
    "Test fraction |theta| > pi:",
    np.mean(np.abs(X_qnn_test.flatten()) > np.pi),
)

# ------------------------------------------------------------
# Convert to torch.float64 tensors for noisy SPSA + Adam training
# ------------------------------------------------------------
# Use .tolist() to avoid relying on torch's NumPy bridge.

X_train_noisy = torch.tensor(
    np.asarray(X_qnn_train, dtype=np.float64).tolist(),
    dtype=torch.float64,
)

y_train_noisy = torch.tensor(
    np.asarray(y_qnn_train, dtype=np.float64).reshape(-1).tolist(),
    dtype=torch.float64,
)

X_val_noisy = torch.tensor(
    np.asarray(X_qnn_val, dtype=np.float64).tolist(),
    dtype=torch.float64,
)

y_val_noisy = torch.tensor(
    np.asarray(y_qnn_val, dtype=np.float64).reshape(-1).tolist(),
    dtype=torch.float64,
)

X_test_noisy = torch.tensor(
    np.asarray(X_qnn_test, dtype=np.float64).tolist(),
    dtype=torch.float64,
)

y_test_noisy = torch.tensor(
    np.asarray(y_qnn_test, dtype=np.float64).reshape(-1).tolist(),
    dtype=torch.float64,
)

print("\nTorch tensors for noisy fixed-depth QNN:")
print("X_train_noisy:", X_train_noisy.shape)
print("y_train_noisy:", y_train_noisy.shape)
print("X_val_noisy:", X_val_noisy.shape)
print("y_val_noisy:", y_val_noisy.shape)
print("X_test_noisy:", X_test_noisy.shape)
print("y_test_noisy:", y_test_noisy.shape)

# ------------------------------------------------------------
# Shape checks
# ------------------------------------------------------------

assert X_train_noisy.ndim == 2
assert X_val_noisy.ndim == 2
assert X_test_noisy.ndim == 2

assert X_train_noisy.shape[1] == n_qubits
assert X_val_noisy.shape[1] == n_qubits
assert X_test_noisy.shape[1] == n_qubits

assert y_train_noisy.ndim == 1
assert y_val_noisy.ndim == 1
assert y_test_noisy.ndim == 1

assert X_train_noisy.shape[0] == y_train_noisy.shape[0]
assert X_val_noisy.shape[0] == y_val_noisy.shape[0]
assert X_test_noisy.shape[0] == y_test_noisy.shape[0]

# ------------------------------------------------------------
# Target diagnostics
# ------------------------------------------------------------
# Current noisy model ends with Tanh(), so predictions are in [-1, 1].

print("\nTarget diagnostics:")
print("y train min:", y_train_noisy.min().item())
print("y train max:", y_train_noisy.max().item())
print("y train mean:", y_train_noisy.mean().item())
print("y train std:", y_train_noisy.std().item())

print("y val min:", y_val_noisy.min().item())
print("y val max:", y_val_noisy.max().item())

print("y test min:", y_test_noisy.min().item())
print("y test max:", y_test_noisy.max().item())

if y_train_noisy.min().item() < -1.05 or y_train_noisy.max().item() > 1.05:
    print(
        "\nWARNING: y values are outside [-1, 1]. "
        "Your model currently ends with Tanh(), so predictions are bounded. "
        "This may still match your classical simulation setup, but keep it in mind."
    )
else:
    print("\ny range looks compatible with final Tanh().")

# ------------------------------------------------------------
# Tiny real-data forward test
# ------------------------------------------------------------

batch_size_test = 1

idx = torch.randint(
    low=0,
    high=X_train_noisy.shape[0],
    size=(batch_size_test,),
)

X_batch_real_test = X_train_noisy[idx]
y_batch_real_test = y_train_noisy[idx]

print("\nTiny real-data test batch:")
print("X_batch_real_test:", X_batch_real_test.shape)
print("y_batch_real_test:", y_batch_real_test.shape)

test_loss_real = evaluate_hybrid_loss(
    model,
    X_batch_real_test,
    y_batch_real_test,
)

print("One-batch real-data noisy hybrid loss:", test_loss_real)

print("\nBlock 10 completed successfully.")

Fixed noisy QNN configuration:
fixed_n_qubits: 6
fixed_n_layers: 1
fixed_n_ansatz_layers: 1

Using CSV file:
/Users/mansigoel/Documents/GitHub/Quantum-Machine-Learning-/SPEI_AllScales_Napak - SPEI_AllScales_Napak.csv

Raw data shapes:
df: (539, 17)
df_train: (434, 17)
df_test: (105, 17)

Checkpoint directory:
/Users/mansigoel/Documents/GitHub/Quantum-Machine-Learning-/data-re upload/experiments/lstm_ae_tuning_latent6_window20
Exists? True

Available checkpoints:
 - run_001_lr_0.01_bs_16_latent_6_win_20.pth
 - run_002_lr_0.01_bs_32_latent_6_win_20.pth
 - run_003_lr_0.01_bs_64_latent_6_win_20.pth
 - run_004_lr_0.05_bs_16_latent_6_win_20.pth
 - run_005_lr_0.05_bs_32_latent_6_win_20.pth
 - run_006_lr_0.05_bs_64_latent_6_win_20.pth
 - run_007_lr_0.001_bs_16_latent_6_win_20.pth
 - run_008_lr_0.001_bs_32_latent_6_win_20.pth
 - run_009_lr_0.001_bs_64_latent_6_win_20.pth
 - run_010_lr_0.005_bs_16_latent_6_win_20.pth
 - run_011_lr_0.005_bs_32_latent_6_win_20.pth
 - run_012_lr_0.005_bs_64_latent_

In [35]:
# ============================================================
# Block 11: Real-data fixed-depth noisy SPSA + Adam training loop
# ============================================================

def evaluate_hybrid_loss_on_subset(
    model,
    X_data,
    y_data,
    subset_size=8,
):
    """
    Evaluate noisy hybrid loss on a small random subset.

    Important:
        Full validation is expensive because every sample requires
        a noisy Qiskit circuit execution.
    """

    model.eval()

    n_samples = X_data.shape[0]

    subset_size = min(subset_size, n_samples)

    idx = torch.randint(
        low=0,
        high=n_samples,
        size=(subset_size,),
    )

    X_subset = X_data[idx]
    y_subset = y_data[idx]

    with torch.no_grad():
        y_pred = model(X_subset)
        loss = loss_fn(y_pred, y_subset)

    return float(loss.detach().cpu().item())


def train_noisy_spsa_adam_fixed_depth(
    model,
    q_params_pl,
    X_train,
    y_train,
    X_val=None,
    y_val=None,
    num_iters=10,
    batch_size=1,
    val_subset_size=8,
    log_every=1,
):
    """
    Fixed-depth noisy QNN training loop.

    Each iteration:
        1. Draw mini-batch from real training data
        2. SPSA update on q_params
        3. Adam update on classical head
        4. Log train batch loss and small-subset validation loss

    This does NOT sweep depth.
    This uses the fixed global n_layers and n_ansatz_layers already defined.
    """

    history = {
        "iter": [],
        "train_batch_loss": [],
        "val_subset_loss": [],
        "q_norm": [],
        "head_weight_norm": [],
        "head_bias_norm": [],
    }

    n_train = X_train.shape[0]

    for k in range(num_iters):
        # ----------------------------------------------------
        # Mini-batch
        # ----------------------------------------------------
        idx = torch.randint(
            low=0,
            high=n_train,
            size=(batch_size,),
        )

        X_batch = X_train[idx]
        y_batch = y_train[idx]

        # ----------------------------------------------------
        # 1. SPSA update for quantum parameters
        # ----------------------------------------------------
        q_params_pl = pennylane_spsa_step_qparams(
            model,
            q_params_pl,
            X_batch,
            y_batch,
        )

        # ----------------------------------------------------
        # 2. Adam update for classical head
        # ----------------------------------------------------
        train_batch_loss = adam_step_classical_head(
            model,
            X_batch,
            y_batch,
        )

        # ----------------------------------------------------
        # Logging
        # ----------------------------------------------------
        if k % log_every == 0:
            q_norm = torch.norm(model.q_params.detach()).item()
            head_weight_norm = torch.norm(model.classical[0].weight.detach()).item()
            head_bias_norm = torch.norm(model.classical[0].bias.detach()).item()

            if X_val is not None and y_val is not None:
                val_subset_loss = evaluate_hybrid_loss_on_subset(
                    model,
                    X_val,
                    y_val,
                    subset_size=val_subset_size,
                )
            else:
                val_subset_loss = np.nan

            history["iter"].append(k)
            history["train_batch_loss"].append(train_batch_loss)
            history["val_subset_loss"].append(val_subset_loss)
            history["q_norm"].append(q_norm)
            history["head_weight_norm"].append(head_weight_norm)
            history["head_bias_norm"].append(head_bias_norm)

            print(
                f"iter={k:03d} | "
                f"train_batch_loss={train_batch_loss:.6f} | "
                f"val_subset_loss={val_subset_loss:.6f} | "
                f"q_norm={q_norm:.6f} | "
                f"head_w_norm={head_weight_norm:.6f} | "
                f"head_b_norm={head_bias_norm:.6f}"
            )

    return q_params_pl, history

In [36]:
# ============================================================
# Block 11A: Run real-data noisy SPSA + Adam smoke test
# ============================================================

# Reinitialize model from scratch for the real-data run.
# We are NOT transferring the ideal Adam-trained model yet.

model = HybridModelNoisy().double()

q_params_pl = init_q_params_for_spsa(model)

head_optimizer = torch.optim.Adam(
    model.classical.parameters(),
    lr=1e-3,
)

print("Fresh noisy hybrid model initialized.")
print("q_params_pl shape:", q_params_pl.shape)

q_params_pl, noisy_history_smoke = train_noisy_spsa_adam_fixed_depth(
    model=model,
    q_params_pl=q_params_pl,
    X_train=X_train_noisy,
    y_train=y_train_noisy,
    X_val=X_val_noisy,
    y_val=y_val_noisy,
    num_iters=10,
    batch_size=1,
    val_subset_size=8,
    log_every=1,
)

print("\nReal-data noisy SPSA + Adam smoke test completed.")

Fresh noisy hybrid model initialized.
q_params_pl shape: (1, 1, 6, 3)
iter=000 | train_batch_loss=0.107836 | val_subset_loss=0.639179 | q_norm=0.045249 | head_w_norm=0.663778 | head_b_norm=0.190675
iter=001 | train_batch_loss=0.155530 | val_subset_loss=0.753400 | q_norm=0.045164 | head_w_norm=0.664402 | head_b_norm=0.189676
iter=002 | train_batch_loss=0.052055 | val_subset_loss=0.714842 | q_norm=0.044204 | head_w_norm=0.665479 | head_b_norm=0.188741
iter=003 | train_batch_loss=0.498920 | val_subset_loss=0.395517 | q_norm=0.044013 | head_w_norm=0.665990 | head_b_norm=0.187793
iter=004 | train_batch_loss=0.561942 | val_subset_loss=0.566144 | q_norm=0.043268 | head_w_norm=0.666763 | head_b_norm=0.186837
iter=005 | train_batch_loss=0.044784 | val_subset_loss=0.608411 | q_norm=0.043429 | head_w_norm=0.667650 | head_b_norm=0.185928
iter=006 | train_batch_loss=0.958362 | val_subset_loss=0.513796 | q_norm=0.043031 | head_w_norm=0.668550 | head_b_norm=0.185000
iter=007 | train_batch_loss=0.7891

In [37]:
# ============================================================
# Block 12: Controlled fixed-depth noisy SPSA + Adam training
# ============================================================

import copy

# ------------------------------------------------------------
# Controlled evaluation on a fixed subset
# ------------------------------------------------------------

def evaluate_hybrid_loss_fixed_subset(
    model,
    X_data,
    y_data,
    fixed_indices,
):
    """
    Evaluate noisy hybrid loss on a fixed subset.

    This is still shot-noisy, but the data points are fixed,
    so the metric is more comparable across iterations.
    """

    model.eval()

    X_subset = X_data[fixed_indices]
    y_subset = y_data[fixed_indices]

    with torch.no_grad():
        y_pred = model(X_subset)
        loss = loss_fn(y_pred, y_subset)

    return float(loss.detach().cpu().item())


# ------------------------------------------------------------
# Controlled training loop
# ------------------------------------------------------------

def train_noisy_spsa_adam_controlled(
    model,
    q_params_pl,
    X_train,
    y_train,
    X_val,
    y_val,
    num_iters=50,
    batch_size=1,
    val_subset_size=8,
    log_every=5,
):
    """
    Controlled fixed-depth noisy training.

    Each iteration:
        1. Sample train mini-batch
        2. SPSA update on q_params
        3. Adam update on classical head
        4. Evaluate on a fixed validation subset every log_every steps

    Returns:
        updated q_params_pl
        history
        best_state_dict
    """

    n_train = X_train.shape[0]
    n_val = X_val.shape[0]

    val_subset_size = min(val_subset_size, n_val)

    # Fixed validation subset for consistent monitoring
    fixed_val_indices = torch.randperm(n_val)[:val_subset_size]

    history = {
        "iter": [],
        "train_batch_loss": [],
        "val_subset_loss": [],
        "q_norm": [],
        "head_weight_norm": [],
        "head_bias_norm": [],
    }

    best_val_loss = float("inf")
    best_state_dict = None
    best_q_params_pl = None
    best_iter = None

    print("Fixed validation subset size:", val_subset_size)
    print("Training iterations:", num_iters)
    print("Batch size:", batch_size)
    print("Logging every:", log_every)

    for k in range(num_iters):
        # ----------------------------------------------------
        # Sample training mini-batch
        # ----------------------------------------------------
        idx = torch.randint(
            low=0,
            high=n_train,
            size=(batch_size,),
        )

        X_batch = X_train[idx]
        y_batch = y_train[idx]

        # ----------------------------------------------------
        # 1. SPSA update for quantum parameters
        # ----------------------------------------------------
        q_params_pl = pennylane_spsa_step_qparams(
            model,
            q_params_pl,
            X_batch,
            y_batch,
        )

        # ----------------------------------------------------
        # 2. Adam update for classical head
        # ----------------------------------------------------
        train_batch_loss = adam_step_classical_head(
            model,
            X_batch,
            y_batch,
        )

        # ----------------------------------------------------
        # Logging and checkpointing
        # ----------------------------------------------------
        if k % log_every == 0 or k == num_iters - 1:
            val_subset_loss = evaluate_hybrid_loss_fixed_subset(
                model,
                X_val,
                y_val,
                fixed_indices=fixed_val_indices,
            )

            q_norm = torch.norm(model.q_params.detach()).item()
            head_weight_norm = torch.norm(model.classical[0].weight.detach()).item()
            head_bias_norm = torch.norm(model.classical[0].bias.detach()).item()

            history["iter"].append(k)
            history["train_batch_loss"].append(train_batch_loss)
            history["val_subset_loss"].append(val_subset_loss)
            history["q_norm"].append(q_norm)
            history["head_weight_norm"].append(head_weight_norm)
            history["head_bias_norm"].append(head_bias_norm)

            improved = val_subset_loss < best_val_loss

            if improved:
                best_val_loss = val_subset_loss
                best_iter = k
                best_state_dict = copy.deepcopy(model.state_dict())
                best_q_params_pl = pnp.array(
                    np.asarray(q_params_pl, dtype=np.float64),
                    requires_grad=True,
                )

            marker = " <-- best" if improved else ""

            print(
                f"iter={k:03d} | "
                f"train_batch_loss={train_batch_loss:.6f} | "
                f"val_subset_loss={val_subset_loss:.6f} | "
                f"q_norm={q_norm:.6f} | "
                f"head_w_norm={head_weight_norm:.6f} | "
                f"head_b_norm={head_bias_norm:.6f}"
                f"{marker}"
            )

    print("\nBest validation subset loss:", best_val_loss)
    print("Best iteration:", best_iter)

    return q_params_pl, history, best_state_dict, best_q_params_pl

In [38]:
# ============================================================
# Block 12A: Run controlled real-data noisy training
# ============================================================

# ------------------------------------------------------------
# Reinitialize everything for a clean fixed-depth experiment
# ------------------------------------------------------------

model = HybridModelNoisy().double()

q_params_pl = init_q_params_for_spsa(model)

# Reinitialize PennyLane SPSA optimizer for a clean schedule
spsa_opt = qml.SPSAOptimizer(
    maxiter=300,
    a=0.02,
    c=0.05,
    A=50,
    alpha=0.602,
    gamma=0.101,
)

# Reinitialize Adam for the classical head
head_optimizer = torch.optim.Adam(
    model.classical.parameters(),
    lr=1e-3,
)

print("Fresh fixed-depth noisy model initialized.")
print("n_qubits:", n_qubits)
print("n_layers:", n_layers)
print("n_ansatz_layers:", n_ansatz_layers)
print("q_params_pl shape:", q_params_pl.shape)

# ------------------------------------------------------------
# Run training
# ------------------------------------------------------------

q_params_pl, noisy_history_controlled, best_state_dict, best_q_params_pl = (
    train_noisy_spsa_adam_controlled(
        model=model,
        q_params_pl=q_params_pl,
        X_train=X_train_noisy,
        y_train=y_train_noisy,
        X_val=X_val_noisy,
        y_val=y_val_noisy,
        num_iters=50,
        batch_size=1,
        val_subset_size=8,
        log_every=5,
    )
)

print("\nControlled noisy SPSA + Adam training completed.")

Fresh fixed-depth noisy model initialized.
n_qubits: 6
n_layers: 1
n_ansatz_layers: 1
q_params_pl shape: (1, 1, 6, 3)
Fixed validation subset size: 8
Training iterations: 50
Batch size: 1
Logging every: 5
iter=000 | train_batch_loss=0.619866 | val_subset_loss=0.965004 | q_norm=0.054687 | head_w_norm=0.506392 | head_b_norm=0.223560 <-- best
iter=005 | train_batch_loss=1.299156 | val_subset_loss=0.953333 | q_norm=0.055450 | head_w_norm=0.502892 | head_b_norm=0.218858 <-- best
iter=010 | train_batch_loss=0.126941 | val_subset_loss=0.943891 | q_norm=0.056823 | head_w_norm=0.498918 | head_b_norm=0.214459 <-- best
iter=015 | train_batch_loss=0.937943 | val_subset_loss=0.933149 | q_norm=0.058838 | head_w_norm=0.494608 | head_b_norm=0.210003 <-- best
iter=020 | train_batch_loss=0.096219 | val_subset_loss=0.923333 | q_norm=0.058039 | head_w_norm=0.490985 | head_b_norm=0.205444 <-- best
iter=025 | train_batch_loss=0.187434 | val_subset_loss=0.913421 | q_norm=0.058088 | head_w_norm=0.487526 | hea

In [39]:
# ============================================================
# Block 13: Replicate original QNN training conditions
#           Fixed noisy QNN, depth = 6, SPSA + Adam
# ============================================================

import copy
import time
from torch.utils.data import DataLoader, TensorDataset

# ------------------------------------------------------------
# Match original ideal notebook conditions
# ------------------------------------------------------------

seed = 42

n_qubits = 6
n_layers = 6              # original best/fixed reupload depth we want now
n_ansatz_layers = 1

qnn_n_epochs = 200
qnn_batch_size = 16
qnn_learning_rate = 0.005

use_output_tanh = True

# Keep shots modest first. Increase to 1024 later if runtime is okay.
shots = 512

set_seed = None

def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_global_seed(seed)

print("Replicated noisy QNN settings:")
print("n_qubits:", n_qubits)
print("n_layers / reupload depth:", n_layers)
print("n_ansatz_layers:", n_ansatz_layers)
print("qnn_n_epochs:", qnn_n_epochs)
print("qnn_batch_size:", qnn_batch_size)
print("classical Adam lr:", qnn_learning_rate)
print("use_output_tanh:", use_output_tanh)
print("shots:", shots)

Replicated noisy QNN settings:
n_qubits: 6
n_layers / reupload depth: 6
n_ansatz_layers: 1
qnn_n_epochs: 200
qnn_batch_size: 16
classical Adam lr: 0.005
use_output_tanh: True
shots: 512


In [40]:
# ============================================================
# Block 13A: Redefine noisy QNode for depth = 6
# ============================================================

@qml.set_shots(shots)
@qml.qnode(dev_noisy, interface=None, diff_method=None)
def circuit_parallel_noisy(q_params, x):
    """
    Noisy 6-qubit data re-uploading circuit.

    q_params shape:
        (n_layers, n_ansatz_layers, n_qubits, 3)

    x shape:
        (n_qubits,)
    """

    for layer in range(n_layers):

        # Data upload
        for q in range(n_qubits):
            qml.RY(x[q], wires=q)

        # Trainable ansatz
        for ansatz_layer in range(n_ansatz_layers):

            for q in range(n_qubits):
                qml.Rot(
                    q_params[layer, ansatz_layer, q, 0],
                    q_params[layer, ansatz_layer, q, 1],
                    q_params[layer, ansatz_layer, q, 2],
                    wires=q,
                )

            # Same as your ideal notebook:
            # nearest-neighbour chain plus ring-closing CNOT
            for q in range(n_qubits - 1):
                qml.CNOT(wires=[q, q + 1])

            if n_qubits > 2:
                qml.CNOT(wires=[n_qubits - 1, 0])

    return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]


# Quick shape test
test_q_params = 0.01 * np.random.randn(
    n_layers,
    n_ansatz_layers,
    n_qubits,
    3,
)

test_x = np.asarray(X_train_noisy[0].detach().cpu().tolist(), dtype=np.float64)

test_q_out = circuit_parallel_noisy(test_q_params, test_x)

print("Noisy depth-6 QNode test successful.")
print("test_q_params shape:", test_q_params.shape)
print("test_x shape:", test_x.shape)
print("test_q_out length:", len(test_q_out))
print("test_q_out:", test_q_out)

Noisy depth-6 QNode test successful.
test_q_params shape: (6, 1, 6, 3)
test_x shape: (6,)
test_q_out length: 6
test_q_out: [array(-0.1484375), array(-0.140625), array(-0.015625), array(-0.06640625), array(0.), array(0.12109375)]


In [41]:
# ============================================================
# Block 13B: Define noisy HybridModel for depth = 6
# ============================================================

class HybridModelNoisy(nn.Module):
    def __init__(self):
        super().__init__()

        self.q_params = nn.Parameter(
            0.01 * torch.randn(
                n_layers,
                n_ansatz_layers,
                n_qubits,
                3,
                dtype=torch.float64,
            )
        )

        if use_output_tanh:
            self.classical = nn.Sequential(
                nn.Linear(n_qubits, 1),
                nn.Tanh(),
            ).double()
        else:
            self.classical = nn.Linear(n_qubits, 1).double()

    def _torch_to_numpy_safe(self, tensor):
        if isinstance(tensor, torch.Tensor):
            return np.asarray(
                tensor.detach().cpu().tolist(),
                dtype=np.float64,
            )

        return np.asarray(tensor, dtype=np.float64)

    def quantum_features(self, x_batch, q_params=None):
        if q_params is None:
            q_params = self.q_params

        q_params_np = self._torch_to_numpy_safe(q_params)

        q_outputs = []

        for x in x_batch:
            x_np = self._torch_to_numpy_safe(x)

            q_out = circuit_parallel_noisy(q_params_np, x_np)

            q_out_list = [float(v) for v in q_out]

            q_out_tensor = torch.tensor(
                q_out_list,
                dtype=torch.float64,
                device=x_batch.device,
            )

            q_outputs.append(q_out_tensor)

        q_outputs = torch.stack(q_outputs)

        return q_outputs

    def forward(self, x_batch, q_params=None):
        q_outputs = self.quantum_features(
            x_batch,
            q_params=q_params,
        )

        # No PyTorch backprop through noisy quantum backend.
        q_outputs = q_outputs.detach()

        y_pred = self.classical(q_outputs)

        return y_pred.squeeze(-1)


# Test model shape
model = HybridModelNoisy().double()

print(model)
print("q_params shape:", model.q_params.shape)

X_test_batch = X_train_noisy[:2]
y_pred_test = model(X_test_batch)

print("Depth-6 model forward test successful.")
print("X_test_batch:", X_test_batch.shape)
print("y_pred_test:", y_pred_test.shape)
print("y_pred_test:", y_pred_test)

HybridModelNoisy(
  (classical): Sequential(
    (0): Linear(in_features=6, out_features=1, bias=True)
    (1): Tanh()
  )
)
q_params shape: torch.Size([6, 1, 6, 3])
Depth-6 model forward test successful.
X_test_batch: torch.Size([2, 6])
y_pred_test: torch.Size([2])
y_pred_test: tensor([0.1581, 0.1695], grad_fn=<SqueezeBackward1>)


In [42]:
# ============================================================
# Block 13C: DataLoaders matching original notebook
# ============================================================

def make_noisy_qnn_loaders(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    batch_size=16,
):
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)
    test_dataset = TensorDataset(X_test, y_test)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    return train_loader, val_loader, test_loader


train_loader_noisy, val_loader_noisy, test_loader_noisy = make_noisy_qnn_loaders(
    X_train=X_train_noisy,
    y_train=y_train_noisy,
    X_val=X_val_noisy,
    y_val=y_val_noisy,
    X_test=X_test_noisy,
    y_test=y_test_noisy,
    batch_size=qnn_batch_size,
)

print("Noisy DataLoaders created.")
print("Train batches:", len(train_loader_noisy))
print("Val batches:", len(val_loader_noisy))
print("Test batches:", len(test_loader_noisy))
print("Train samples:", len(train_loader_noisy.dataset))
print("Val samples:", len(val_loader_noisy.dataset))
print("Test samples:", len(test_loader_noisy.dataset))

Noisy DataLoaders created.
Train batches: 22
Val batches: 4
Test batches: 6
Train samples: 350
Val samples: 64
Test samples: 85


In [43]:
# ============================================================
# Block 13D: SPSA + Adam utilities for depth-6 epoch training
# ============================================================

loss_fn = nn.MSELoss()

def init_q_params_for_spsa(model):
    q_params_list = model.q_params.detach().cpu().tolist()

    q_params_pl = pnp.array(
        q_params_list,
        dtype=np.float64,
        requires_grad=True,
    )

    return q_params_pl


def copy_q_params_to_model(model, q_params_pl):
    q_params_np = np.asarray(
        q_params_pl,
        dtype=np.float64,
    )

    q_params_tensor = torch.tensor(
        q_params_np.tolist(),
        dtype=torch.float64,
        device=model.q_params.device,
    )

    with torch.no_grad():
        model.q_params.copy_(q_params_tensor)


def make_spsa_cost(model, X_batch, y_batch):
    X_batch = X_batch.detach()
    y_batch = y_batch.detach()

    def cost(q_params_candidate):
        model.eval()

        with torch.no_grad():
            y_pred = model(
                X_batch,
                q_params=q_params_candidate,
            )

            loss = loss_fn(
                y_pred,
                y_batch,
            )

        return pnp.array(
            loss.detach().cpu().item(),
            requires_grad=False,
        )

    return cost


def pennylane_spsa_step_qparams(
    model,
    q_params_pl,
    X_batch,
    y_batch,
):
    cost_fn = make_spsa_cost(
        model,
        X_batch,
        y_batch,
    )

    q_params_pl_new = spsa_opt.step(
        cost_fn,
        q_params_pl,
    )

    copy_q_params_to_model(
        model,
        q_params_pl_new,
    )

    return q_params_pl_new


def adam_step_classical_head(
    model,
    X_batch,
    y_batch,
):
    model.train()

    head_optimizer.zero_grad()

    y_pred = model(X_batch)

    loss = loss_fn(
        y_pred,
        y_batch,
    )

    loss.backward()

    head_optimizer.step()

    return float(loss.detach().cpu().item())


def evaluate_loader_noisy(
    model,
    data_loader,
):
    model.eval()

    total_loss = 0.0
    n_total = 0

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            y_pred = model(X_batch)

            loss = loss_fn(
                y_pred,
                y_batch,
            )

            batch_n = X_batch.shape[0]

            total_loss += float(loss.detach().cpu().item()) * batch_n
            n_total += batch_n

    return total_loss / n_total

In [44]:
# ============================================================
# Block 13E: Full epoch-style noisy SPSA + Adam training
# ============================================================

def train_one_noisy_qnn_model_spsa_adam(
    model,
    train_loader,
    val_loader,
    n_epochs=200,
    classical_learning_rate=0.005,
    print_every=50,
    max_train_batches_per_epoch=None,
):
    """
    Noisy equivalent of train_one_qnn_model from the ideal notebook.

    Original ideal training:
        optimizer = Adam(model.parameters(), lr=0.005)

    Noisy training:
        q_params        -> PennyLane SPSA
        classical head  -> Adam(lr=0.005)

    If max_train_batches_per_epoch is None:
        uses all batches, matching original training most closely.

    If max_train_batches_per_epoch is small:
        useful for debugging runtime.
    """

    global spsa_opt
    global head_optimizer

    model = model.double()

    q_params_pl = init_q_params_for_spsa(model)

    total_spsa_steps = n_epochs * len(train_loader)

    spsa_opt = qml.SPSAOptimizer(
        maxiter=total_spsa_steps,
        a=0.02,
        c=0.05,
        A=50,
        alpha=0.602,
        gamma=0.101,
    )

    head_optimizer = torch.optim.Adam(
        model.classical.parameters(),
        lr=classical_learning_rate,
    )

    train_losses = []
    val_losses = []

    best_val_loss = float("inf")
    best_model_state = None
    best_q_params_pl = None
    best_epoch = None

    print("Starting noisy epoch-style training")
    print("n_epochs:", n_epochs)
    print("train batches per epoch:", len(train_loader))
    print("batch_size:", train_loader.batch_size)
    print("total SPSA steps:", total_spsa_steps)
    print("max_train_batches_per_epoch:", max_train_batches_per_epoch)

    start_time = time.time()

    for epoch in range(n_epochs):
        model.train()

        total_train_loss = 0.0
        total_train_n = 0

        for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
            if (
                max_train_batches_per_epoch is not None
                and batch_idx >= max_train_batches_per_epoch
            ):
                break

            # 1. SPSA quantum update
            q_params_pl = pennylane_spsa_step_qparams(
                model,
                q_params_pl,
                X_batch,
                y_batch,
            )

            # 2. Adam classical-head update
            batch_loss = adam_step_classical_head(
                model,
                X_batch,
                y_batch,
            )

            batch_n = X_batch.shape[0]

            total_train_loss += batch_loss * batch_n
            total_train_n += batch_n

        avg_train_loss = total_train_loss / total_train_n

        # Full validation, matching original notebook logic.
        avg_val_loss = evaluate_loader_noisy(
            model,
            val_loader,
        )

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            best_q_params_pl = pnp.array(
                np.asarray(q_params_pl, dtype=np.float64),
                requires_grad=True,
            )
            best_epoch = epoch + 1

        if print_every is not None and print_every > 0:
            if (epoch + 1) % print_every == 0 or epoch == 0:
                elapsed = time.time() - start_time

                print(
                    f"Epoch [{epoch + 1}/{n_epochs}] "
                    f"Train Loss: {avg_train_loss:.6f} "
                    f"Val Loss: {avg_val_loss:.6f} "
                    f"Best Val: {best_val_loss:.6f} "
                    f"Best Epoch: {best_epoch} "
                    f"Elapsed: {elapsed:.2f} sec"
                )

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        copy_q_params_to_model(model, best_q_params_pl)

    elapsed_time = time.time() - start_time

    history = {
        "train_loss": train_losses,
        "val_loss": val_losses,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "elapsed_time_sec": elapsed_time,
        "best_q_params_pl": best_q_params_pl,
    }

    return model, history

In [45]:
# ============================================================
# Block 13F: Run replicated noisy training
# ============================================================

set_global_seed(seed)

model = HybridModelNoisy().double()

start_time = time.time()

model, noisy_depth6_history = train_one_noisy_qnn_model_spsa_adam(
    model=model,
    train_loader=train_loader_noisy,
    val_loader=val_loader_noisy,
    n_epochs=qnn_n_epochs,
    classical_learning_rate=qnn_learning_rate,
    print_every=50,

    # For true replication, keep this as None.
    # For a runtime test, set this to 2 or 5 first.
    max_train_batches_per_epoch=2,
)

elapsed_time = time.time() - start_time

print("\nDepth-6 noisy training completed.")
print("Best val loss scaled:", noisy_depth6_history["best_val_loss"])
print("Best epoch:", noisy_depth6_history["best_epoch"])
print("Elapsed time:", elapsed_time)

Starting noisy epoch-style training
n_epochs: 200
train batches per epoch: 22
batch_size: 16
total SPSA steps: 4400
max_train_batches_per_epoch: 2
Epoch [1/200] Train Loss: 0.783796 Val Loss: 0.806539 Best Val: 0.806539 Best Epoch: 1 Elapsed: 29.61 sec
Epoch [50/200] Train Loss: 0.155014 Val Loss: 0.251454 Best Val: 0.251454 Best Epoch: 50 Elapsed: 1517.45 sec


KeyboardInterrupt: 